In [ ]:
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup

# =========================
# CONFIG
# =========================
URL_FILE = "tvnet_faktomats_urls.csv"

SOURCE_NAME = "TVNET (Faktomats)"
DEFAULT_LABEL = 1  # misinformation / fact-check

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "lv-LV,lv;q=0.9,en;q=0.8",
    "Referer": "https://www.tvnet.lv/"
}

SLEEP_BETWEEN_ARTICLES = 0.25


# =========================
# HELPERS
# =========================
def safe_get(url, timeout=25):
    response = requests.get(url, headers=HEADERS, timeout=timeout)
    response.raise_for_status()
    return response


def scrape_tvnet_article(url):
    soup = BeautifulSoup(safe_get(url).content, "html.parser")

    # Title
    title_tag = soup.find("h1", class_="article__headline") or soup.find("h1")
    title = title_tag.get_text(strip=True) if title_tag else ""

    # Publication date
    publication_date = ""
    date_node = soup.find(attrs={"itemprop": "datePublished"})
    if date_node and date_node.has_attr("content"):
        publication_date = date_node["content"][:10]

    if not publication_date:
        meta = soup.find("meta", attrs={"property": "article:published_time"})
        if meta and meta.has_attr("content"):
            publication_date = meta["content"][:10]

    # Full text
    content_divs = soup.find_all("div", class_="article-body__item--htmlElement")
    if not content_divs:
        content_divs = soup.select("div.article-body, div.article__body, article")

    full_text = " ".join(
        div.get_text(" ", strip=True)
        for div in content_divs
        if div.get_text(strip=True)
    )
    full_text = " ".join(full_text.split())

    return title, full_text, publication_date


# =========================
# MAIN: SCRAPE FROM URL LIST
# =========================
urls = pd.read_csv(URL_FILE, encoding="utf-8")["url"].dropna().unique().tolist()
total_urls = len(urls)

print(f"Starting TVNET Faktomats scraping: {total_urls} URLs\n")

records = []

for i, url in enumerate(urls, start=1):
    print(f"Scraping {i}/{total_urls}: {url}")

    try:
        title, full_text, publication_date = scrape_tvnet_article(url)

        records.append({
            "title": title,
            "full_text": full_text,
            "source": SOURCE_NAME,
            "publication_date": publication_date,
            "topic": "faktomats",
            "class_label": DEFAULT_LABEL,
            "url": url
        })

    except Exception as e:
        print(f"Error scraping {url}: {e}")

    time.sleep(SLEEP_BETWEEN_ARTICLES)


# =========================
# SAVE RAW DATASET
# =========================
df_raw = pd.DataFrame(records)

print(f"\nScraped records (raw): {len(df_raw)}")

RAW_FILE = "tvnet_faktomats_raw.csv"
df_raw.to_csv(RAW_FILE, index=False, encoding="utf-8")

print(f"Saved raw CSV: {RAW_FILE}")
print("Done.")

Starting TVNET Faktomats scraping: 140 URLs

Scraping 1/140: https://www.tvnet.lv/8280294/faktomats-vai-svaigpiens-patiesam-ir-daudz-veseligaks-par-parasto-pienu
Scraping 2/140: https://www.tvnet.lv/8283562/faktomats-ne-kailgliemezus-latvija-neieveda-ar-merki-iznicinat-vietejo-lauksaimniecibu
Scraping 3/140: https://www.tvnet.lv/8287613/faktomats-skaidra-nauda-pazudis-no-aprites-miti-un-patiesiba-par-digitalo-eiro
Scraping 4/140: https://www.tvnet.lv/8293209/faktomats-vai-vacijas-kanclers-mercs-medi-polarlacus-kanada
Scraping 5/140: https://www.tvnet.lv/8300996/faktomats-rusofobija-kremla-galvenais-politiskais-mits
Scraping 6/140: https://www.tvnet.lv/8302600/faktomats-vai-arzemju-pilsoni-rietumos-tiesam-izdara-vairak-noziegumu-neka-vietejie
Scraping 7/140: https://www.tvnet.lv/8308394/faktomats-vai-krievija-ir-izstradajusi-vakcinu-pret-melanomu-parbaudam-olgas-petkevicas-izteikumus
Scraping 8/140: https://www.tvnet.lv/8310970/faktomats-ne-miljardierim-zelenskim-nav-pils-anglija-un-jau

1. Raw data inspection

In [1]:
import pandas as pd
import re
from collections import Counter

FILE = "tvnet_faktomats_raw.csv"

df = pd.read_csv(FILE, encoding="utf-8")

print("\nROWS:", len(df))
print("COLUMNS:", df.columns.tolist())


# --------------------------------------------------
# TITLES
# --------------------------------------------------

print("\nFIRST 5 TITLES:")
for i, t in enumerate(df["title"].fillna("").astype(str).head(5), start=1):
    print(f"{i}. {t}")

print("\nLAST 5 TITLES:")
for i, t in enumerate(df["title"].fillna("").astype(str).tail(5), start=1):
    print(f"{i}. {t}")


# --------------------------------------------------
# TEXT LENGTH
# --------------------------------------------------

lengths = df["full_text"].fillna("").astype(str).str.len()

print("\nTEXT LENGTH SUMMARY:")
print("Min:", int(lengths.min()))
print("Median:", int(lengths.median()))
print("Max:", int(lengths.max()))


# --------------------------------------------------
# REPEATED SENTENCES (BOILERPLATE DETECTION)
# --------------------------------------------------

print("\nTOP 40 REPEATED SENTENCES:")

sentences = []

for text in df["full_text"].fillna("").astype(str):

    parts = re.split(r'(?<=[\.\!\?])\s+', text)

    for s in parts:
        s = s.strip()
        if len(s) >= 40:
            sentences.append(s)

top_sent = Counter(sentences).most_common(40)

for s, c in top_sent:
    print(f"{c:>4}  {s[:250]}")


# --------------------------------------------------
# WORD FREQUENCY
# --------------------------------------------------

def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())


all_words = []

for text in df["full_text"].fillna("").astype(str):
    all_words.extend(tokenize(text))

print("\nTOP 50 WORDS:")

for w, c in Counter(all_words).most_common(50):
    print(f"{c:>4}  {w}")


# --------------------------------------------------
# BIGRAMS
# --------------------------------------------------

bigrams = []

for text in df["full_text"].fillna("").astype(str):

    tokens = tokenize(text)

    for i in range(len(tokens) - 1):
        bigrams.append(tokens[i] + " " + tokens[i+1])

print("\nTOP 40 BIGRAMS:")

for b, c in Counter(bigrams).most_common(40):
    print(f"{c:>4}  {b}")


# --------------------------------------------------
# TRIGRAMS
# --------------------------------------------------

trigrams = []

for text in df["full_text"].fillna("").astype(str):

    tokens = tokenize(text)

    for i in range(len(tokens) - 2):
        trigrams.append(tokens[i] + " " + tokens[i+1] + " " + tokens[i+2])

print("\nTOP 40 TRIGRAMS:")

for t, c in Counter(trigrams).most_common(40):
    print(f"{c:>4}  {t}")


# --------------------------------------------------
# FACT CHECK CUE FAMILIES
# --------------------------------------------------

cue_patterns = {
    "faktu pārbaud*": r"faktu\s+pārbaud\w*",
    "maldin*": r"maldin\w*",
    "nepaties*": r"nepaties\w*",
    "paties*": r"paties\w*",
    "apgalvoj*": r"apgalvoj\w*",
    "secin*": r"secin\w*",
    "kontekst*": r"kontekst\w*",
    "dezinform*": r"dezinform\w*",
    "propagand*": r"propagand\w*",
}

print("\nFACT-CHECK CUE FAMILY COUNTS (TEXT):")

for name, pat in cue_patterns.items():

    count = df["full_text"].fillna("").str.contains(
        pat, case=False, regex=True
    ).sum()

    print(f"{count:>4}  {name}")


print("\nFACT-CHECK CUE FAMILY COUNTS (TITLES):")

for name, pat in cue_patterns.items():

    count = df["title"].fillna("").str.contains(
        pat, case=False, regex=True
    ).sum()

    print(f"{count:>4}  {name}")


# --------------------------------------------------
# TITLE PREFIXES BEFORE COLON
# --------------------------------------------------

prefixes = []

for t in df["title"].fillna("").astype(str):

    m = re.match(r"^\s*([^:]{1,60}):", t)

    if m:
        prefixes.append(m.group(1).strip())

print("\nTOP TITLE PREFIXES BEFORE COLON:")

for p, c in Counter(prefixes).most_common(30):
    print(f"{c:>4}  {p}")


# --------------------------------------------------
# SOURCE WORD CONTEXTS
# --------------------------------------------------

SOURCE_WORDS = ["tvnet", "faktomats", "faktomāts"]

print("\nTOP CONTEXTS FOR SOURCE WORDS:")

for word in SOURCE_WORDS:

    print(f"\n--- {word.upper()} ---")

    contexts = []

    for text in df["full_text"].fillna("").astype(str):

        for m in re.finditer(word, text, flags=re.IGNORECASE):

            start = max(0, m.start() - 60)
            end = min(len(text), m.end() + 60)

            contexts.append(text[start:end])

    for c in contexts[:20]:
        print(" ", c.replace("\n", " "))


ROWS: 140
COLUMNS: ['title', 'full_text', 'source', 'publication_date', 'topic', 'class_label', 'url']

FIRST 5 TITLES:
1. Faktomāts⟩Vai svaigpiens patiešām ir daudz veselīgāks par “parasto” pienu?(4)
2. Faktomāts⟩Nē, kailgliemežus Latvijā neieveda ar mērķi iznīcināt vietējo lauksaimniecību
3. Faktomāts⟩Skaidra nauda pazudīs no aprites? Mīti un patiesība par digitālo eiro(6)
4. FAKTOMĀTS⟩Vai Vācijas kanclers Mercs medī polārlāčus Kanādā?(5)
5. Faktomāts⟩Rusofobija – Kremļa galvenais politiskais mīts

LAST 5 TITLES:
1. FAKTOMĀTS⟩Kāds labums no Stambulas konvencijas(68)
2. FAKTOMĀTS⟩“Zelta standarts” medicīnā: ko tas nozīmē un kāpēc zinātnieki ceļ trauksmi ASV(13)
3. FAKTOMĀTS⟩Putina ģenerāļi atkal lielās ar izdomātām uzvarām kaujas laukā. Diktators atkal visam tic
4. FAKTOMĀTS⟩Vai Latvija nav spējusi panākt izņēmumu migrācijas kvotām?(3)
5. FAKTOMĀTS⟩Šveicē aizliegta mamogrāfija?(3)

TEXT LENGTH SUMMARY:
Min: 354
Median: 4301
Max: 17577

TOP 40 REPEATED SENTENCES:
  33  Pilna atbildība

tep 1 inspection was used to separate source-identifying and boilerplate elements from verdict-signalling expressions. Repeated project labels, funding notices, outlet references, and technical noise were assigned to source cleaning, while lexical cues directly associated with fact-checking judgement, such as faktu pārbaud-, maldin-, nepaties-, paties-, apgalvoj-, and secin-, were assigned to the masking stage.

2. Cleaning

In [3]:
%%writefile clean_tvnet_faktomats.py
from __future__ import annotations

import re
import html
import pandas as pd

IN_FILE = "tvnet_faktomats_raw.csv"
OUT_FILE = "tvnet_faktomats_clean.csv"

MIN_TEXT_LEN = 300

# --------------------------------------------------
# Title cleanup
# --------------------------------------------------
# Remove leading project prefix:
# Faktomāts⟩...
# FAKTOMĀTS⟩...
TITLE_PREFIX_PATTERNS = [
    r'^\s*Faktomāts\s*⟩\s*',
    r'^\s*FAKTOMĀTS\s*⟩\s*',
]

# Remove trailing comment counts:
# ...(4)
# ...(68)
TITLE_TRAILING_COUNT_PATTERN = re.compile(r"\s*\(\d+\)\s*$")

# --------------------------------------------------
# Boilerplate / funding / disclaimer / navigation noise
# --------------------------------------------------
REMOVE_PATTERNS = [
    r'Projektu\s+[“"]?Faktomāts[”"]?\s+jeb\s+[“"]?Truth\s+Flash[”"]?\s+atbalsta\s+Eiropas\s+Mediju\s+un\s+informācijas\s+fonds\s+\(EMIF\)\.?',
    r'Pilna\s+atbildība\s+par\s+jebkādu\s+EMIF\s+atbalstīto\s+projektu\s+saturu\s+gulstas\s+uz\s+šī\s+satura\s+autoru\s*\(?-?iem\)?[,]?\s+un\s+šis\s+saturs\s+var\s+nesakrist\s+ar\s+EMIF\s+un\s+fonda\s+partneru\s+[–-]\s+Kalust[ao]?\s+Gulbenkiana\s+fonda\s+un\s+Eiropas\s+Universitātes\s+institūta\s+[–-]\s+pozīciju\.?',
    r'Pilna\s+atbildība\s+par\s+jebkādu\s+EMIF\s+atbalstīto\s+projektu\s+saturu\s+gulstas\s+uz\s+šī\s+satura\s+autoru\s*\(?-?iem\)?[,]?\s+un\s+šis\s+saturs\s+var\s+nesakrist\s+ar\s+EMIF\s+un\s+fonda\s+partneru.*',
    r'RUS\s+TVNET.*',
    r'Символичное\s+дело\s+ЕСПЧ\s+против\s+Латвии.*',
]

# Cut everything from the first matched tail marker to the end
TAIL_MARKERS = [
    r'Projektu\s+[“"]?Faktomāts[”"]?\s+jeb\s+[“"]?Truth\s+Flash[”"]?\s+atbalsta\s+Eiropas\s+Mediju\s+un\s+informācijas\s+fonds',
    r'Pilna\s+atbildība\s+par\s+jebkādu\s+EMIF\s+atbalstīto\s+projektu\s+saturu',
    r'RUS\s+TVNET',
]

# --------------------------------------------------
# Source / project branding
# Important: Faktomāts is source/project branding here,
# so it belongs to cleaning, not masking.
# --------------------------------------------------
SOURCE_PATTERNS = [
    r'\bTVNET\s+faktu\s+pārbaudes\s+projekts\b',
    r'\bTVNET\s+projekts\b',
    r'\bTVNET\s+projektā\b',
    r'\bTVNET\s+GRUPAS\s+faktu\s+pārbaudes\s+projekts\b',
    r'\bTVNET\+\b',
    r'\bTVNET\b',
    r'\bportālam\s+TVNET\b',
    r'\bportālā\s+TVNET\b',
    r'\bportāls\s+TVNET\b',

    r'[“"]\s*Faktomāts\s*[”"]',
    r'"\s*Faktomāts\s*"',
    r'\bFaktomāts\b',
    r'\bFAKTOMĀTS\b',
    r'\bTruth\s+Flash\b',

    r'@\w*DelfiLV\b',  # just in case social snippets survived in similar texts
]

COMPILED_REMOVE_PATTERNS = [
    re.compile(pat, flags=re.IGNORECASE | re.UNICODE | re.DOTALL)
    for pat in REMOVE_PATTERNS + SOURCE_PATTERNS
]


def normalize_text(text: object) -> str:
    if pd.isna(text):
        return ""

    text = str(text)
    text = html.unescape(text)

    # Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Normalize punctuation
    text = text.replace("\u201c", '"').replace("\u201d", '"')
    text = text.replace("\u2018", "'").replace("\u2019", "'")
    text = text.replace("\u2013", "-").replace("\u2014", "-")
    text = text.replace("\u00a0", " ").replace("\u200b", "")

    # Normalize whitespace
    text = re.sub(r"\r\n|\r", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)

    return text.strip()


def cut_from_first_marker(text: str, markers: list[str]) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""

    positions = []

    for pat in markers:
        m = re.search(pat, text, flags=re.IGNORECASE | re.UNICODE | re.DOTALL)
        if m:
            positions.append(m.start())

    if not positions:
        return text

    return text[:min(positions)].strip()


def clean_title(title: object) -> str:
    title = normalize_text(title)

    for pat in TITLE_PREFIX_PATTERNS:
        title = re.sub(pat, "", title, flags=re.IGNORECASE | re.UNICODE)

    title = TITLE_TRAILING_COUNT_PATTERN.sub("", title)
    title = re.sub(r"\s+", " ", title).strip()

    return title


def clean_full_text(text: object) -> str:
    text = normalize_text(text)

    # First cut long tails
    text = cut_from_first_marker(text, TAIL_MARKERS)

    # Then remove inline boilerplate and branding
    for pattern in COMPILED_REMOVE_PATTERNS:
        text = pattern.sub(" ", text)

    # Cleanup leftovers
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    text = re.sub(r"([,.;:!?]){2,}", r"\1", text)
    text = re.sub(r"\(\s*\)", " ", text)
    text = re.sub(r"\[\s*\]", " ", text)
    text = re.sub(r"\s*-\s*-\s*", " - ", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"^[,.;:\-\s]+", "", text)

    return text.strip()


# --------------------------------------------------
# Load
# --------------------------------------------------
df = pd.read_csv(IN_FILE, encoding="utf-8")

rows_before = len(df)
columns_before = df.columns.tolist()

# --------------------------------------------------
# Create cleaned columns
# --------------------------------------------------
df["title_clean"] = df["title"].apply(clean_title)
df["full_text_clean"] = df["full_text"].apply(clean_full_text)
df["text_length_clean"] = df["full_text_clean"].fillna("").astype(str).str.len()

# --------------------------------------------------
# Remove empty / too short texts
# --------------------------------------------------
df = df[df["full_text_clean"].fillna("").astype(str).str.strip().str.len() >= MIN_TEXT_LEN].copy()

rows_after = len(df)
removed_rows = rows_before - rows_after

df = df.reset_index(drop=True)

# --------------------------------------------------
# Save
# --------------------------------------------------
df.to_csv(OUT_FILE, index=False, encoding="utf-8")

# --------------------------------------------------
# Report
# --------------------------------------------------
columns_after = df.columns.tolist()
new_columns = [c for c in columns_after if c not in columns_before]

print("Saved:", OUT_FILE)
print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Removed rows:", removed_rows)

print("\nColumns before:")
print(columns_before)

print("\nColumns after:")
print(columns_after)

print("\nNew columns added:")
print(new_columns if new_columns else "None")

Overwriting clean_tvnet_faktomats.py


In [4]:
!python clean_tvnet_faktomats.py

Saved: tvnet_faktomats_clean.csv
Rows before: 140
Rows after: 140
Removed rows: 0

Columns before:
['title', 'full_text', 'source', 'publication_date', 'topic', 'class_label', 'url']

Columns after:
['title', 'full_text', 'source', 'publication_date', 'topic', 'class_label', 'url', 'title_clean', 'full_text_clean', 'text_length_clean']

New columns added:
['title_clean', 'full_text_clean', 'text_length_clean']


3. Masking verdict clues

In [5]:
%%writefile mask_tvnet_faktomats.py
from __future__ import annotations

import re
import pandas as pd

IN_FILE = "tvnet_faktomats_clean.csv"
OUT_FILE = "tvnet_faktomats_masked.csv"
AUDIT_FILE = "tvnet_faktomats_mask_audit.csv"

MASK_TOKEN = "[MASK]"

# --------------------------------------------------
# Title masking
# Faktomāts itself was already removed in Step 2.
# Here we mask verdict / fact-check cue words only.
# --------------------------------------------------
TITLE_MASK_PATTERNS = [
    r'^\s*Nav\s+taisn(?:ība|i)\s*[:\-–—]?\s*',
    r'^\s*Trūkst\s+kontekst\w*\s*[:\-–—]?\s*',
    r'^\s*Secin\w*\s*[:\-–—]?\s*',
    r'^\s*Apgalvoj\w*\s*[:\-–—]?\s*',
    r'^\s*Nepaties\w*\s*[:\-–—]?\s*',
    r'^\s*Maldin\w*\s*[:\-–—]?\s*',
]

# --------------------------------------------------
# Text masking
# --------------------------------------------------
TEXT_MASK_PATTERNS = [
    r'\bnav\s+taisn(?:ība|i)\b',
    r'\btrūkst\s+kontekst\w*\b',
    r'\bfaktu\s+pārbaud\w*\b',
    r'\bnepaties\w*\b',
    r'\bmaldin\w*\b',
    r'\bapgalvoj\w*\b',
    r'\bsecin\w*\b',
    r'\bkontekst\w*\b',
    r'\bpaties\w*\b',
    r'\bdezinform\w*\b',
    r'\bpropagand\w*\b',
]

COMPILED_TITLE_MASK_PATTERNS = [
    re.compile(p, flags=re.IGNORECASE | re.UNICODE)
    for p in TITLE_MASK_PATTERNS
]

COMPILED_TEXT_MASK_PATTERNS = [
    re.compile(p, flags=re.IGNORECASE | re.UNICODE)
    for p in TEXT_MASK_PATTERNS
]


def safe_text(text: object) -> str:
    if pd.isna(text):
        return ""
    return str(text).strip()


def cleanup_masked_text(text: str) -> str:
    text = re.sub(r'(\[MASK\]\s*){2,}', MASK_TOKEN + " ", text)
    text = re.sub(r'\[MASK\]\s*[,;:]+\s*\[MASK\]', MASK_TOKEN, text)
    text = re.sub(r'\[MASK\]\s*[-–—:]\s*\[MASK\]', MASK_TOKEN, text)
    text = re.sub(r'^\s*[,;:.\-–—]+\s*', "", text)
    text = re.sub(r'\(\s*\[MASK\]\s*\)', MASK_TOKEN, text)
    text = re.sub(r'\[\s*\]', ' ', text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def mask_title(text: object) -> str:
    text = safe_text(text)

    for pattern in COMPILED_TITLE_MASK_PATTERNS:
        text = pattern.sub(MASK_TOKEN + " ", text)

    for pattern in COMPILED_TEXT_MASK_PATTERNS:
        text = pattern.sub(MASK_TOKEN, text)

    text = cleanup_masked_text(text)
    text = re.sub(r"^\[MASK\]\s*[:\-–—]\s*", MASK_TOKEN + " ", text)

    return text.strip()


def mask_full_text(text: object) -> str:
    text = safe_text(text)

    for pattern in COMPILED_TEXT_MASK_PATTERNS:
        text = pattern.sub(MASK_TOKEN, text)

    text = cleanup_masked_text(text)
    return text.strip()


df = pd.read_csv(IN_FILE, encoding="utf-8")

rows_before = len(df)

# ----------------------------------------
# Apply masking
# ----------------------------------------
df["title_masked"] = df["title_clean"].apply(mask_title)
df["full_text_masked"] = df["full_text_clean"].apply(mask_full_text)

# ----------------------------------------
# Mask diagnostics
# ----------------------------------------
df["title_changed_by_masking"] = df["title_clean"] != df["title_masked"]
df["text_changed_by_masking"] = df["full_text_clean"] != df["full_text_masked"]

df["title_mask_count"] = df["title_masked"].fillna("").astype(str).str.count(r"\[MASK\]")
df["text_mask_count"] = df["full_text_masked"].fillna("").astype(str).str.count(r"\[MASK\]")

# ----------------------------------------
# Save main masked file
# ----------------------------------------
df.to_csv(OUT_FILE, index=False, encoding="utf-8")

# ----------------------------------------
# Save audit file
# ----------------------------------------
audit_df = df[
    df["title_changed_by_masking"] | df["text_changed_by_masking"]
][[
    "title",
    "title_clean",
    "title_masked",
    "full_text",
    "full_text_clean",
    "full_text_masked",
    "title_mask_count",
    "text_mask_count",
]].copy()

audit_df.to_csv(AUDIT_FILE, index=False, encoding="utf-8")

# ----------------------------------------
# Report
# ----------------------------------------
print("Saved:", OUT_FILE)
print("Saved audit:", AUDIT_FILE)
print("Rows:", rows_before)

print("\nMask summary:")
print({
    "title_changed_rows": int(df["title_changed_by_masking"].sum()),
    "text_changed_rows": int(df["text_changed_by_masking"].sum()),
    "total_title_masks": int(df["title_mask_count"].sum()),
    "total_text_masks": int(df["text_mask_count"].sum()),
})

print("\nNew columns added:")
print([
    "title_masked",
    "full_text_masked",
    "title_changed_by_masking",
    "text_changed_by_masking",
    "title_mask_count",
    "text_mask_count",
])

Writing mask_tvnet_faktomats.py


In [6]:
!python mask_tvnet_faktomats.py

Saved: tvnet_faktomats_masked.csv
Saved audit: tvnet_faktomats_mask_audit.csv
Rows: 140

Mask summary:
{'title_changed_rows': 17, 'text_changed_rows': 126, 'total_title_masks': 20, 'total_text_masks': 708}

New columns added:
['title_masked', 'full_text_masked', 'title_changed_by_masking', 'text_changed_by_masking', 'title_mask_count', 'text_mask_count']


4. Remove mask

In [7]:
%%writefile remove_mask_tvnet_faktomats.py
from __future__ import annotations

import re
import pandas as pd

IN_FILE = "tvnet_faktomats_masked.csv"
OUT_FILE = "tvnet_faktomats_nomask.csv"

MASK_PATTERN = r"\[MASK\]"


def remove_mask_tokens(text: object) -> str:
    if pd.isna(text):
        return ""

    text = str(text)

    # remove mask token
    text = re.sub(MASK_PATTERN, " ", text, flags=re.IGNORECASE)

    # remove awkward leading punctuation
    text = re.sub(r"^\s*[,;:.\-–—]+\s*", "", text)

    # clean awkward punctuation fragments after mask removal
    text = re.sub(r"\s*[-–—:]\s*(?=[,.;!?)]|$)", " ", text)
    text = re.sub(r"([.!?])\s*[,;:.\-–—]+\s*", r"\1 ", text)

    # remove empty brackets
    text = re.sub(r"\(\s*\)", " ", text)
    text = re.sub(r"\[\s*\]", " ", text)

    # normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


df = pd.read_csv(IN_FILE, encoding="utf-8")

rows_before = len(df)

before_title_masks = df["title_masked"].fillna("").astype(str).str.count(MASK_PATTERN).sum()
before_text_masks = df["full_text_masked"].fillna("").astype(str).str.count(MASK_PATTERN).sum()

df["title_masked"] = df["title_masked"].apply(remove_mask_tokens)
df["full_text_masked"] = df["full_text_masked"].apply(remove_mask_tokens)

after_title_masks = df["title_masked"].fillna("").astype(str).str.count(MASK_PATTERN).sum()
after_text_masks = df["full_text_masked"].fillna("").astype(str).str.count(MASK_PATTERN).sum()

df.to_csv(OUT_FILE, index=False, encoding="utf-8")

print("Saved:", OUT_FILE)
print("Rows:", rows_before)
print("\nMask removal summary:")
print({
    "title_masks_before": int(before_title_masks),
    "text_masks_before": int(before_text_masks),
    "title_masks_after": int(after_title_masks),
    "text_masks_after": int(after_text_masks),
})

Writing remove_mask_tvnet_faktomats.py


In [8]:
!python remove_mask_tvnet_faktomats.py

Saved: tvnet_faktomats_nomask.csv
Rows: 140

Mask removal summary:
{'title_masks_before': 20, 'text_masks_before': 708, 'title_masks_after': 0, 'text_masks_after': 0}


 5. Source level dedup

In [9]:
%%writefile dedup_tvnet_faktomats.py
from __future__ import annotations

import pandas as pd
import hashlib

IN_FILE = "tvnet_faktomats_nomask.csv"
OUT_FILE = "tvnet_faktomats_dedup.csv"


def normalize_text(text: str) -> str:
    if pd.isna(text):
        return ""

    text = str(text).lower().strip()
    text = " ".join(text.split())

    return text


def text_hash(text: str) -> str:
    return hashlib.md5(text.encode("utf-8")).hexdigest()


df = pd.read_csv(IN_FILE, encoding="utf-8")

rows_before = len(df)

# --------------------------------------------------
# URL duplicates
# --------------------------------------------------
if "url" in df.columns:
    df["is_duplicate_url"] = df.duplicated(subset=["url"], keep="first")
else:
    df["is_duplicate_url"] = False


# --------------------------------------------------
# TEXT duplicates
# --------------------------------------------------
df["text_norm"] = df["full_text_masked"].apply(normalize_text)
df["text_hash"] = df["text_norm"].apply(text_hash)

df["is_duplicate_text"] = df.duplicated(subset=["text_hash"], keep="first")


# --------------------------------------------------
# Report
# --------------------------------------------------
dup_url = int(df["is_duplicate_url"].sum())
dup_text = int(df["is_duplicate_text"].sum())

# --------------------------------------------------
# Drop helper columns
# --------------------------------------------------
df = df.drop(columns=["text_norm", "text_hash"])


# --------------------------------------------------
# Save
# --------------------------------------------------
df.to_csv(OUT_FILE, index=False, encoding="utf-8")

print("Saved:", OUT_FILE)

print("\nRows:", rows_before)

print("\nDuplicates summary:")
print({
    "duplicate_urls": dup_url,
    "duplicate_texts": dup_text
})

print("\nColumns added:")
print(["is_duplicate_url", "is_duplicate_text"])

Writing dedup_tvnet_faktomats.py


In [10]:
!python dedup_tvnet_faktomats.py

Saved: tvnet_faktomats_dedup.csv

Rows: 140

Duplicates summary:
{'duplicate_urls': 0, 'duplicate_texts': 0}

Columns added:
['is_duplicate_url', 'is_duplicate_text']


6. Source level audit

In [11]:
%%writefile audit_tvnet_faktomats.py
from __future__ import annotations

import pandas as pd
import re
from collections import Counter

FILE = "tvnet_faktomats_dedup.csv"

df = pd.read_csv(FILE, encoding="utf-8")

print("ROWS:", len(df))
print("COLUMNS:", df.columns.tolist())

# --------------------------------------------------
# 1) Text length summary
# --------------------------------------------------
print("\nTEXT LENGTH SUMMARY:")
lengths = df["full_text_masked"].fillna("").astype(str).str.len()
print("Min:", int(lengths.min()))
print("Median:", float(lengths.median()))
print("Max:", int(lengths.max()))

# --------------------------------------------------
# 2) Top repeated sentences
# --------------------------------------------------
print("\nTOP 40 REPEATED SENTENCES:")

all_sentences = []

for text in df["full_text_masked"].fillna("").astype(str):
    sentences = re.split(r'(?<=[\.\!\?])\s+', text)

    for sentence in sentences:
        sentence = sentence.strip()
        if len(sentence) >= 40:
            all_sentences.append(sentence)

sentence_counts = Counter(all_sentences)

for sentence, count in sentence_counts.most_common(40):
    print(f"{count:>4}  {sentence[:250]}")

# --------------------------------------------------
# 3) Exact keyword checks in text
# --------------------------------------------------
print("\nEXACT KEYWORD CHECKS (TEXT):")

TEXT_KEYWORDS = [
    "tvnet",
    "faktomāts",
    "faktomats",
    "truth flash",
    "emif",
    "nav taisnība",
    "trūkst konteksta",
    "secinājums",
    "apgalvojums",
    "maldina",
    "nepatiesa",
    "faktu pārbaude",
]

text_series = df["full_text_masked"].fillna("").astype(str)

for k in TEXT_KEYWORDS:
    count = text_series.str.contains(k, case=False, regex=False).sum()
    print(f"{k:>20} : {int(count)}")

# --------------------------------------------------
# 4) Exact keyword checks in titles
# --------------------------------------------------
print("\nEXACT KEYWORD CHECKS (TITLES):")

title_series = df["title_masked"].fillna("").astype(str)

for k in TEXT_KEYWORDS:
    count = title_series.str.contains(k, case=False, regex=False).sum()
    print(f"{k:>20} : {int(count)}")

# --------------------------------------------------
# 5) Contexts for remaining source / project words
# --------------------------------------------------
print("\nTOP CONTEXTS FOR REMAINING SOURCE WORDS:")

SOURCE_PATTERNS = {
    "tvnet": r'\bTVNET(?:\+)?\b',
    "faktomāts": r'\bFaktomāts\b',
    "faktomats": r'\bFaktomats\b',
    "truth flash": r'\bTruth\s+Flash\b',
    "emif": r'\bEMIF\b',
}

for label, pattern in SOURCE_PATTERNS.items():
    contexts = []

    for text in text_series:
        for match in re.finditer(pattern, text, flags=re.IGNORECASE):
            start = max(match.start() - 50, 0)
            end = min(match.end() + 100, len(text))
            context = text[start:end].strip()
            contexts.append(context)

    if contexts:
        print(f"\n--- {label.upper()} ---")
        context_counts = Counter(contexts)
        for context, count in context_counts.most_common(20):
            print(f"{count:>4}  {context}")

# --------------------------------------------------
# 6) Duplicate summary
# --------------------------------------------------
if "is_duplicate_url" in df.columns or "is_duplicate_text" in df.columns:
    print("\nDUPLICATE SUMMARY:")
    if "is_duplicate_url" in df.columns:
        print("Duplicate URLs:", int(df["is_duplicate_url"].sum()))
    if "is_duplicate_text" in df.columns:
        print("Duplicate texts:", int(df["is_duplicate_text"].sum()))

# --------------------------------------------------
# 7) Masking summary
# --------------------------------------------------
if "title_mask_count" in df.columns and "text_mask_count" in df.columns:
    print("\nMASKING SUMMARY:")
    print("Rows with title masks:", int((df["title_mask_count"] > 0).sum()))
    print("Rows with text masks:", int((df["text_mask_count"] > 0).sum()))
    print("Total title masks:", int(df["title_mask_count"].sum()))
    print("Total text masks:", int(df["text_mask_count"].sum()))

# --------------------------------------------------
# 8) Empty checks
# --------------------------------------------------
print("\nEMPTY TEXTS:")
print(int((df["full_text_masked"].fillna("").astype(str).str.strip() == "").sum()))

print("\nEMPTY TITLES:")
print(int((df["title_masked"].fillna("").astype(str).str.strip() == "").sum()))

Writing audit_tvnet_faktomats.py


In [12]:
!python audit_tvnet_faktomats.py

ROWS: 140
COLUMNS: ['title', 'full_text', 'source', 'publication_date', 'topic', 'class_label', 'url', 'title_clean', 'full_text_clean', 'text_length_clean', 'title_masked', 'full_text_masked', 'title_changed_by_masking', 'text_changed_by_masking', 'title_mask_count', 'text_mask_count', 'is_duplicate_url', 'is_duplicate_text']

TEXT LENGTH SUMMARY:
Min: 354
Median: 4018.0
Max: 17047

TOP 40 REPEATED SENTENCES:
   5  jūnijā Latvijā Otrdien Latvijā dzēsti septiņi kūlas ugunsgrēki Ārvalstīs Atjauno dzelzceļa satiksmi starp Ķīnas un Ziemeļkorejas galvaspilsētām Ārvalstīs Ukraiņu droni veikuši triecienu rūpnīcai Toljati (1) Ārvalstīs Pie AAE krastiem neidentificēts š
   3  jūnijā Latvijā Otrdien Latvijā dzēsti septiņi kūlas ugunsgrēki Ārvalstīs Atjauno dzelzceļa satiksmi starp Ķīnas un Ziemeļkorejas galvaspilsētām Ārvalstīs Ukraiņu droni veikuši triecienu rūpnīcai Toljati (1) Ārvalstīs Pie AAE krastiem neidentificēts š
   2  ASV dzimšanas pilsonība tika ieviesta tikai vergu bērniem, nevis i